In [15]:
import os
import pandas as pd
import glob
from tqdm import tqdm
import sys
sys.path.append('..')  # Adds parent directory to sys.path
from utils import StructureToHeteroGraph, StructureToTripleGraphs
import pickle

In [ ]:
swappped_abag_list = [os.path.basename(f) for f in glob.glob('../../GraPPI_data/PDB/AbEpiTope/*pdb')]

In [17]:
swappped_abag_list

['1nca_ag_N_ab_L_H_swap_1wej_ag_F_ab_L_H.pdb',
 '4u0r_ag_A_ab_C_B_swap_1fsk_ag_D_ab_E_F.pdb',
 '4jlr_ag_S_ab_L_H_swap_4u0r_ag_A_ab_C_B.pdb',
 '1iqd_ag_C_ab_A_B_swap_4u0r_ag_A_ab_C_B.pdb',
 '4jlr_ag_S_ab_L_H_swap_5usl_ag_X_ab_L_H.pdb',
 '1vfb_ag_C_ab_A_B_swap_7dfa_ag_A_ab_L_H.pdb',
 '2jel_ag_P_ab_L_H_swap_5otj_ag_D_ab_A_B.pdb',
 '7vyr_ag_C_ab_B_A_swap_3qum_ag_Q_ab_C_D.pdb',
 '5otj_ag_D_ab_A_B_swap_3qum_ag_Q_ab_C_D.pdb',
 '1nsn_ag_S_ab_L_H_swap_1iqd_ag_C_ab_A_B.pdb',
 '4u0r_ag_A_ab_C_B_swap_1nca_ag_N_ab_L_H.pdb',
 '1wej_ag_F_ab_L_H_swap_1vfb_ag_C_ab_A_B.pdb',
 '1nsn_ag_S_ab_L_H_swap_3qum_ag_Q_ab_C_D.pdb',
 '2vir_ag_C_ab_A_B_swap_1nsn_ag_S_ab_L_H.pdb',
 '1bj1_ag_V_W_ab_L_H_swap_1wej_ag_F_ab_L_H.pdb',
 '1jps_ag_T_ab_L_H_swap_4u0r_ag_A_ab_C_B.pdb',
 '4jlr_ag_S_ab_L_H_swap_3qum_ag_Q_ab_C_D.pdb',
 '5otj_ag_D_ab_A_B_swap_1wej_ag_F_ab_L_H.pdb',
 '2vir_ag_C_ab_A_B_swap_1nca_ag_N_ab_L_H.pdb',
 '7z4t_ag_I_ab_L_H_swap_1iqd_ag_C_ab_A_B.pdb',
 '1jps_ag_T_ab_L_H_swap_1nsn_ag_S_ab_L_H.pdb',
 '1nsn_ag_S

In [18]:
# load and reset index of df
#wt_df = pd.read_pickle('./saved_tables/updated_unmutant_complex_table.pkl').reset_index(drop=True)
wt_df = pd.read_csv('./saved_tables/wt_table.csv')
wt_df

,PDB,Receptor Chains,Ligand Chains,paths,affinity
0,1A22,B,A,../../PDB/SKEMPI_v2.0/1A22.pdb,12.622671
1,1A4Y,B,A,../../PDB/SKEMPI_v2.0/1A4Y.pdb,20.658646
2,1ACB,I,E,../../PDB/SKEMPI_v2.0/1ACB.pdb,14.567624
3,1AHW,C,"A, B",../../PDB/SKEMPI_v2.0/1AHW.pdb,11.547342
4,1AK4,D,A,../../PDB/SKEMPI_v2.0/1AK4.pdb,6.606704
...,...,...,...,...,...
3093,5E94,"A, B",G,../../PDB/updated_pdb/5E94.pdb,11.621462
3094,5EIV,"A, G, H","I, J, K",../../PDB/updated_pdb/5EIV.pdb,6.203861
3095,5IBK,"A, B",C,../../PDB/updated_pdb/5IBK.pdb,9.756143
3096,6GWC,"A, B",C,../../PDB/updated_pdb/6GWC.pdb,8.956734


In [6]:
AbEpiTope_pdbs_lst = set([pdb_file.split('_')[0].lower() for pdb_file in swappped_abag_list])
wt_pdb_lst = set([pdb.lower() for pdb in wt_df['PDB'].tolist()])
common_pdbs = AbEpiTope_pdbs_lst.intersection(wt_pdb_lst)
common_pdbs, len(common_pdbs)

({'1bj1',
  '1e6j',
  '1fsk',
  '1iqd',
  '1jps',
  '1nca',
  '1nsn',
  '1vfb',
  '1wej',
  '2jel',
  '2vir',
  '4jlr',
  '5otj',
  '5usl'},
 14)

PDB	KD(M), paths, affinity, Structure Method, Ligand Chains, Receptor Chains

In [19]:
swapped_abag_pdb_df = {'PDB':[], 'Receptor Chains':[], 'Ligand Chains':[], 'paths':[], 'affinity':[]}

for pdb_file in swappped_abag_list:
    pdb_content = pdb_file.split('.')[0]
    part1,part2 = pdb_content.split('_swap_')
    # ag of part 1 and ab of part will form the swapped complex
    pdb_ids = part1.split('_ag_')[0]+'_'+part2.split('_ag_')[0]
    # Extract ligand chain letters (couldbe multiple letters) between _ag_ and _ab_ of part1
    ori_rc_ids = ((part1.split('_ag_')[1]).split('_ab_')[0]).split('_')
    # Extract receptor chain letters (could be multiple letters) after _ab_ of part2
    ori_lc_ids = (part2.split('_ab_')[1]).split('_')
    # rename letters in ori_rc_ids and ori_lc_ids to letters starting from A
    rc_ids = [chr(ord('A') + i) for i in range(len(ori_rc_ids))]
    lc_ids = [chr(ord('A') + i) for i in range(len(ori_rc_ids), len(ori_rc_ids) + len(ori_lc_ids))]
    # PDB,Receptor Chains,Ligand Chains,paths,affinity
    swapped_abag_pdb_df['PDB'].append(pdb_ids)
    swapped_abag_pdb_df['Receptor Chains'].append(','.join(rc_ids))
    swapped_abag_pdb_df['Ligand Chains'].append(','.join(lc_ids))
    swapped_abag_pdb_df['paths'].append('../../PDB/AbEpiTope/'+pdb_file)
    swapped_abag_pdb_df['affinity'].append(0.0)
swapped_abag_pdb_df = pd.DataFrame(swapped_abag_pdb_df)

In [20]:
swapped_abag_pdb_df.head(10)

,PDB,Receptor Chains,Ligand Chains,paths,affinity
0,1nca_1wej,A,"B,C",../../PDB/AbEpiTope/1nca_ag_N_ab_L_H_swap_1wej...,0.0
1,4u0r_1fsk,A,"B,C",../../PDB/AbEpiTope/4u0r_ag_A_ab_C_B_swap_1fsk...,0.0
2,4jlr_4u0r,A,"B,C",../../PDB/AbEpiTope/4jlr_ag_S_ab_L_H_swap_4u0r...,0.0
3,1iqd_4u0r,A,"B,C",../../PDB/AbEpiTope/1iqd_ag_C_ab_A_B_swap_4u0r...,0.0
4,4jlr_5usl,A,"B,C",../../PDB/AbEpiTope/4jlr_ag_S_ab_L_H_swap_5usl...,0.0
5,1vfb_7dfa,A,"B,C",../../PDB/AbEpiTope/1vfb_ag_C_ab_A_B_swap_7dfa...,0.0
6,2jel_5otj,A,"B,C",../../PDB/AbEpiTope/2jel_ag_P_ab_L_H_swap_5otj...,0.0
7,7vyr_3qum,A,"B,C",../../PDB/AbEpiTope/7vyr_ag_C_ab_B_A_swap_3qum...,0.0
8,5otj_3qum,A,"B,C",../../PDB/AbEpiTope/5otj_ag_D_ab_A_B_swap_3qum...,0.0
9,1nsn_1iqd,A,"B,C",../../PDB/AbEpiTope/1nsn_ag_S_ab_L_H_swap_1iqd...,0.0


In [21]:
swapped_abag_pdb_df.to_csv('./saved_tables/AbEpiTope_table.csv', index=False)

In [ ]:
save_path_hg = '../../curated_db/swapped_abag_sthg_8A'
failed_pdbs = []
for ind, row_of_df in tqdm(swapped_abag_pdb_df.iterrows(),total=len(swapped_abag_pdb_df)):
    filename = row_of_df['PDB']#row_of_df["paths"].split("/")[-1].split(".")[0]
    # Check if both output files already exist
    hg_output_path = f'{save_path_hg}/{filename}.pkl'  
    # Process heterograph if not exists
    if not os.path.exists(hg_output_path):
        sthg = StructureToHeteroGraph()
        sthg.process_pdb(row_of_df, root_path='')
        sthg.get_hetero_graph()
        with open(hg_output_path, 'wb') as f:
            pickle.dump(sthg, f)

100%|██████████| 272/272 [1:29:19<00:00, 19.70s/it]
